# 02 — Feature Store: member features ingested to SageMaker Feature Store

**Team Update 3.1 steps 4–5.** Ingests the temporally-split training table
(`aai-540-g1/processed/training/training_table.csv`, built by
`src/build_training_data.py`) into a SageMaker Feature Store feature group,
then proves the offline-store → Athena dataset-extraction path that the
training notebook consumes.

| Step | Where | Verify (📸) |
|---|---|---|
| Feature group create + ingest | this notebook | `get_record` spot-checks |
| Offline store sync | this notebook | Athena split-count query |

**Run order:** top to bottom in SageMaker Studio. If the SDK was already
imported before cell 2 ran, restart the kernel (Kernel → Restart) and re-run.
The offline store syncs asynchronously — the final query may need a re-run
after a few minutes (exactly like Lab 3.1).

## 1. Setup

Pin the SDK: v3 removed the Feature Store APIs this notebook uses (the
`FeatureDefinitions, value: 0` and `sagemaker.session` errors).

In [ ]:
%pip install -q "sagemaker<3"

import boto3
import sagemaker
from sagemaker.session import Session

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client("sagemaker", region_name=region)
featurestore_runtime = boto_session.client("sagemaker-featurestore-runtime", region_name=region)

sess = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)
try:
    role = sagemaker.get_execution_role()  # Studio / notebook instance
except Exception:
    # local run (personal account): reuse the newest SageMaker execution role
    iam = boto3.client("iam", region_name=region)
    candidates = [r for r in iam.list_roles()["Roles"]
                  if r["RoleName"].startswith("AmazonSageMaker-ExecutionRole")]
    if not candidates:
        raise RuntimeError("no SageMaker execution role found - create one in IAM")
    newest = sorted(candidates, key=lambda r: r["CreateDate"])[-1]
    account = boto3.client("sts").get_caller_identity()["Account"]
    role = f"arn:aws:iam::{account}:role/{newest['RoleName']}"
print(f"role={role}")
bucket = sess.default_bucket()
prefix = "aai-540-g1"

print(f"role={role}\nbucket={bucket}\nregion={region}")

## 2. Load the training table from S3

In [ ]:
import pandas as pd

boto3.client("s3").download_file(
    bucket, f"{prefix}/processed/training/training_table.csv", "training_table.csv"
)
df = pd.read_csv("training_table.csv")

# Feature Store record identifiers are strings; event_time must be a numeric
# epoch (the tz-aware datetime dtype fails load_feature_definitions - Lab 3.1).
df["person_id"] = df["person_id"].astype(str)
df["event_time"] = pd.Timestamp.now(tz="UTC").timestamp()

COLUMN_ORDER = [c for c in df.columns if c != "event_time"] + ["event_time"]
df = df[COLUMN_ORDER]
print(f"{len(df):,} rows x {df.shape[1]} cols")
df.head(3)

## 3. Create the feature group and ingest

One row per member: 20 history-window features + `risk_class` (label) +
`split` (train/validation/test/production) + `event_time`.

In [ ]:
import time
from sagemaker.feature_store.feature_group import FeatureGroup

FEATURE_GROUP_NAME = "aai-540-g1-member-features"

feature_group = FeatureGroup(name=FEATURE_GROUP_NAME, sagemaker_session=sess)
feature_group.load_feature_definitions(data_frame=df)  # required before create

feature_group.create(
    s3_uri=f"s3://{bucket}/{prefix}/features",
    record_identifier_name="person_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True,
)

status = feature_group.describe().get("FeatureGroupStatus")
while status == "Creating":
    print("Waiting for Feature Group Creation")
    time.sleep(5)
    status = feature_group.describe().get("FeatureGroupStatus")
assert status == "Created", f"unexpected status: {status}"
print(f"FeatureGroup {FEATURE_GROUP_NAME} created.")

In [ ]:
feature_group.ingest(data_frame=df, max_workers=8, wait=True)
print(f"Ingested {len(df):,} member records.")

## 4. 📸 Online store spot-checks

`get_record` for three members — one from the train split, one from
production, one from test (verify against the printed rows).

In [ ]:
for split_name in ["train", "production", "test"]:
    pid = df.loc[df["split"] == split_name, "person_id"].iloc[0]
    record = featurestore_runtime.get_record(
        FeatureGroupName=FEATURE_GROUP_NAME,
        RecordIdentifierValueAsString=str(pid),
    )
    feats = {f["FeatureName"]: f["ValueAsString"] for f in record.get("Record", [])}
    expected = df[df["person_id"] == str(pid)].iloc[0]
    ok = feats.get("risk_class") == expected["risk_class"] and \
         feats.get("split") == split_name and \
         int(float(feats.get("total_visits", -1))) == int(expected["total_visits"])
    print(f"person {pid} [{split_name}] risk={feats.get('risk_class')} "
          f"visits={feats.get('total_visits')} -> {'MATCH' if ok else 'MISMATCH'}")

## 5. 📸 Offline store dataset extraction (Athena)

The offline store lands asynchronously (Kinesis Firehose buffer — usually
5–15 minutes). Re-run the count query until `n_total` equals 82,624, then
screenshot: it is the proof that Feature Store → training-dataset extraction
works for all four splits.

In [ ]:
athena = boto_session.client("athena", region_name=region)
WORKGROUP = "aai540_g1"

def run_query(sql):
    qid = athena.start_query_execution(QueryString=sql, WorkGroup=WORKGROUP)[
        "QueryExecutionId"]
    while True:
        state = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        if state["State"] == "SUCCEEDED":
            return athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
        if state["State"] in ("FAILED", "CANCELLED"):
            raise RuntimeError(state.get("StateChangeReason"))
        time.sleep(2)

# offline-store Glue table: <feature-group-name>-<creation-epoch>
fg_table = next(
    t for t in boto_session.client("glue", region_name=region).get_tables(
        DatabaseName="sagemaker_featurestore"
    )["TableList"] if FEATURE_GROUP_NAME in t["Name"]
)["Name"]

rows = run_query(
    f'SELECT split, risk_class, COUNT(*) AS n FROM "sagemaker_featurestore"."{fg_table}" '
    f"GROUP BY split, risk_class ORDER BY split, risk_class"
)
for row in rows:
    print(" | ".join(c.get("VarCharValue", "") for c in row["Data"]))

total = run_query(f'SELECT COUNT(*) AS n FROM "sagemaker_featurestore"."{fg_table}"')
print("n_total:", total[1]["Data"][0].get("VarCharValue"), "(target 82,624)")

## Optional cleanup (after grading)

Feature Store charges accrue while the online store exists. After the project
is graded, delete the feature group:

```python
sagemaker_client.delete_feature_group(FeatureGroupName=FEATURE_GROUP_NAME)
```